In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
from sklearn.svm import SVR
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn import datasets

In [3]:
df = datasets.load_diabetes(as_frame = True).frame

In [4]:
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [5]:
df.shape

(442, 11)

In [6]:
X = df.drop("target", axis  = 1)
y = df["target"]

In [7]:
X.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42)

In [9]:
y_scaler = StandardScaler()

y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1,1)).ravel()# ravel is used to revert back the 2D to 1D
y_test_scaled = y_scaler.transform(y_test.values.reshape(-1,1)).ravel()

In [10]:
model = SVR()

model.fit(X_train,y_train_scaled)

SVR()

In [11]:
y_test_pred_scaled = model.predict(X_test)
y_train_pred_scaled = model.predict(X_train)

In [12]:
print("test r2:",r2_score(y_test_scaled, y_test_pred_scaled))
print("train r2:",r2_score(y_train_scaled, y_train_pred_scaled))

test r2: 0.48844443151651906
train r2: 0.6596361676267712


there is some overfitting cur for test we are only getting 48 and for train it's 65 the difference between them is more so overfitting

In [13]:
y_pred_scaled = model.predict(X_test)

In [14]:
print("r2:",r2_score(y_test_scaled, y_pred_scaled))

r2: 0.48844443151651906


In [15]:
# Linear
model = SVR(kernel = "linear")

model.fit(X_train,y_train_scaled)
y_pred_scaled = model.predict(X_test)
print("r1:", r2_score(y_test_scaled, y_pred_scaled))

r1: 0.4433761323833776


In [16]:
# Linear
model = SVR(kernel = "linear")

model.fit(X_train,y_train_scaled)
y_test_pred_scaled = model.predict(X_test)
y_train_pred_scaled = model.predict(X_train)
print("test r2:",r2_score(y_test_scaled, y_test_pred_scaled))
print("train r2:",r2_score(y_train_scaled, y_train_pred_scaled))

test r2: 0.4433761323833776
train r2: 0.45191229982475245


its better than before cuz its diff is less

In [17]:
# polynomial
model = SVR(kernel = "poly")

model.fit(X_train,y_train_scaled)
y_pred_scaled = model.predict(X_test)
print("r1:", r2_score(y_test_scaled, y_pred_scaled))

r1: 0.24203771038107735


In [18]:
# sigmoid
model = SVR(kernel = "sigmoid")

model.fit(X_train,y_train_scaled)
y_pred_scaled = model.predict(X_test)
print("r1:", r2_score(y_test_scaled, y_pred_scaled))

r1: -15.316808189576822


# Hyperparameter tuning using GridSearchCV

In [19]:
from sklearn.model_selection import GridSearchCV

In [20]:
param_grid = {
    "C":[1, 2, 5, 10, 50, 100],
    "kernel": ["rbf", "linear"],
    "epsilon": [0.01, 0.1, 0.2, 0.3, 0.5]
}

In [21]:
svr = SVR()

grid_search = GridSearchCV(svr, param_grid, scoring = "r2", cv=5)

grid_search.fit(X_train, y_train_scaled)

GridSearchCV(cv=5, estimator=SVR(),
             param_grid={'C': [1, 2, 5, 10, 50, 100],
                         'epsilon': [0.01, 0.1, 0.2, 0.3, 0.5],
                         'kernel': ['rbf', 'linear']},
             scoring='r2')

In [22]:
print("best params -", grid_search.best_params_)

best params - {'C': 10, 'epsilon': 0.1, 'kernel': 'linear'}


In [23]:
best_model = SVR(kernel="linear", C=10, epsilon=0.1)

best_model.fit(X_train,y_train_scaled)

y_test_pred_scaled = best_model.predict(X_test)
y_train_pred_scaled = best_model.predict(X_train)

print("test r2:",r2_score(y_test_scaled, y_test_pred_scaled))
print("train r2:",r2_score(y_train_scaled, y_train_pred_scaled))

test r2: 0.47444183250401095
train r2: 0.5151066486918875


In [24]:
from sklearn.svm import LinearSVR

model = SVR(C=10, epsilon=0.1, max_iter=5000)

model.fit(X_train,y_train_scaled)

y_test_pred_scaled = model.predict(X_test)
y_train_pred_scaled = model.predict(X_train)

print("test r2:",r2_score(y_test_scaled, y_test_pred_scaled))
print("train r2:",r2_score(y_train_scaled, y_train_pred_scaled))

test r2: 0.21097310184556106
train r2: 0.8336324884196327
